# Sales bookings and fulfillment

Inspect the order-line and shipment-line grains and the shared dimensions. The two baseline orders should produce two booking rows, 11 pumps, and ₹140,000.

Run from the `gold/` directory after setting `silver_manifest` in `gold_settings.json`.

In [1]:
from pathlib import Path

from IPython.display import display
import pandas as pd

from gold_pipeline import build, publish

SETTINGS_PATH = Path("gold_settings.json")
gold = build(SETTINGS_PATH, stage="bookings")
print("Silver release", gold.manifest["model_run_id"])
print("Cutoff", gold.settings["cutoff"])
display(pd.DataFrame([{"table": name, "rows": len(rows)} for name, rows in gold.tables.items()]))

Silver release 409a707f49d90b112334f8bb4bfd27423a7f3f48f7039eecdff871dddee256e5
Cutoff 2026-12-16


,table,rows
0,dim_legal_entity,2
1,dim_customer,5
2,dim_product,5
3,dim_date,6
4,dim_currency,1
5,fact_sales_order_booking,8
6,dim_warehouse,3
7,fact_shipment_line,5


In [2]:
baseline = {"P-S1", "D-S1"}
display(pd.DataFrame([
    row for row in gold.records("fact_sales_order_booking")
    if row["order_reference"] in baseline
]))

,id,sales_order_id,source_system,order_id,order_line_id,order_reference,legal_entity_id,booking_date,currency,booked_amount,source_qty,source_uom,booked_qty,booked_uom,line_count,booking_date_key,customer_key,product_key,legal_entity_key,currency_key
0,"[""parent"",""SAL-ORD-2026-00001"",""vqgdj6mnv1""]","[""parent"",""SAL-ORD-2026-00001""]",erpnext,SAL-ORD-2026-00001,vqgdj6mnv1,P-S1,parent,2026-11-02,INR,100000.000000000000,10.000000000000,Nos,10.000000000000,each,1,date:2026-11-02,customer:G-CUST-001,product:G-PROD-001,legal_entity:parent,currency:INR
1,"[""dealer"",""D-S1"",""D-L1""]","[""dealer"",""D-S1""]",dealer-sql,D-S1,D-L1,D-S1,dealer,2026-11-05,INR,40000.000000000000,1.000000000000,Each,1.000000000000,each,1,date:2026-11-05,customer:G-CUST-001,product:G-PROD-001,legal_entity:dealer,currency:INR


In [3]:
for name in ("dim_customer", "dim_product", "dim_legal_entity", "fact_shipment_line"):
    print(name)
    display(pd.DataFrame(gold.records(name)).head(20))

dim_customer


,key,business_id
0,customer:G-CUST-001,G-CUST-001
1,customer:G-CUST-002,G-CUST-002
2,customer:G-CUST-005,G-CUST-005
3,customer:G-CUST-006,G-CUST-006
4,customer:__unknown__,__unknown__


dim_product


,key,business_id
0,product:G-PROD-001,G-PROD-001
1,product:G-PROD-002,G-PROD-002
2,product:G-PROD-005,G-PROD-005
3,product:G-PROD-006,G-PROD-006
4,product:__unknown__,__unknown__


dim_legal_entity


,key,business_id,name
0,legal_entity:parent,parent,OTC Parent Demo
1,legal_entity:dealer,dealer,Acquired dealer


fact_shipment_line


,id,shipment_id,shipment_line_id,sales_order_line_id,shipped_qty,shipped_uom,line_count,shipment_date_key,legal_entity_key,warehouse_key,customer_key,product_key
0,"[""parent"",""MAT-DN-2026-00001"",""vqnei0r1v0""]","[""parent"",""MAT-DN-2026-00001""]",vqnei0r1v0,"[""parent"",""SAL-ORD-2026-00001"",""vqgdj6mnv1""]",6.000000000000,Nos,1,date:2026-11-03,legal_entity:parent,"warehouse:[""parent"",""P-WH1 - OPD""]",customer:G-CUST-001,product:G-PROD-001
1,"[""parent"",""MAT-DN-2026-00002"",""vqtf30brbp""]","[""parent"",""MAT-DN-2026-00002""]",vqtf30brbp,"[""parent"",""SAL-ORD-2026-00001"",""vqgdj6mnv1""]",4.000000000000,Nos,1,date:2026-11-05,legal_entity:parent,"warehouse:[""parent"",""P-WH2 - OPD""]",customer:G-CUST-001,product:G-PROD-001
2,"[""parent"",""MAT-DN-2026-00003"",""brlofkakoo""]","[""parent"",""MAT-DN-2026-00003""]",brlofkakoo,"[""parent"",""SAL-ORD-2026-00006"",""brbeshsujo""]",2.000000000000,Nos,1,date:2026-11-15,legal_entity:parent,"warehouse:[""parent"",""P-WH1 - OPD""]",customer:G-CUST-001,product:G-PROD-001
3,"[""dealer"",""D-BRONZE-R2-SHIP"",""D-BRONZE-R2-SHIP""]","[""dealer"",""D-BRONZE-R2-SHIP""]",D-BRONZE-R2-SHIP,"[""dealer"",""D-BRONZE-R2-ORDER"",""D-BRONZE-R2-LINE""]",2.000000000000,Each,1,date:__unknown__,legal_entity:dealer,"warehouse:[""dealer"",""D-WH1""]",customer:G-CUST-001,product:G-PROD-001
4,"[""dealer"",""D-DEL1"",""D-DEL1""]","[""dealer"",""D-DEL1""]",D-DEL1,"[""dealer"",""D-S1"",""D-L1""]",1.000000000000,Each,1,date:__unknown__,legal_entity:dealer,"warehouse:[""dealer"",""D-WH1""]",customer:G-CUST-001,product:G-PROD-001


In [4]:
display(pd.DataFrame(gold.checks))
display(pd.DataFrame(gold.exceptions))

,rule,expected,actual,passed
0,bookings_preserve_eligible_line_count,8,8,True
1,bookings_preserve_eligible_line_amount,220000.000000000000,220000.000000000000,True
2,baseline_booking_rows,2,2,True
3,baseline_distinct_orders,2,2,True
4,baseline_booking_inr,140000,140000.000000000000,True
5,baseline_shared_customer_product,1,1,True
6,baseline_pumps_each,11,11.000000000000,True


,rule,source_id
0,booking_not_eligible,"[""parent"",""SAL-ORD-2026-00002"",""vqk0j2tpqq""]"
1,booking_not_eligible,"[""parent"",""SAL-ORD-2026-00003"",""vqltad6jqq""]"
2,booking_not_eligible,"[""parent"",""SAL-ORD-2026-00004"",""vqlfquq375""]"
3,booking_not_eligible,"[""parent"",""SAL-ORD-2026-00005"",""vqm83vc86c""]"
